1. Load PyHealth/sample data
2. Build Co-Occurrence Matrix
3. Convert matrix to GloveDataset dataloader that returns i, j, counts, weights
4. Create Keep Model
5. Pass data loader and model to trainer

In [ ]:
from collections import defaultdict, Counter
import networkx as nx
import numpy as np
from pyhealth.datasets import OMOPDataset
from pyhealth.models import KeepEmbedding, N2V
from pyhealth.trainer import Trainer
import warnings
import sys
import torch
from torch.utils.data import Dataset, DataLoader

# Add examples to path to import builder
sys.path.insert(0, '/path/to/examples')  # Will be updated below

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

In [ ]:
class GloveDataset(Dataset):
    def __init__(self, cooc_matrix, num_words, x_max, alpha):
        super(GloveDataset, self).__init__()
        self.data = []
        for i in range(cooc_matrix.shape[0]):
            for j in range(cooc_matrix.shape[1]):
                if cooc_matrix[i, j] > 0:
                    self.data.append((i, j, cooc_matrix[i, j]))
        self.cooc_matrix = cooc_matrix
        self.num_words = num_words
        self.x_max = x_max
        self.alpha = alpha

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        i, j, count = self.data[idx]
        weight = (count / self.x_max) ** self.alpha if count < self.x_max else 1.0
        return torch.tensor(i), torch.tensor(j), torch.tensor(count).float(), torch.tensor(weight).float()
    
def get_code_and_ancestors(graph, code):
    # Start with the code itself
    codes_set = {code}
    
    # Add all ancestors if code exists in graph
    if code in graph:
        ancestors = nx.ancestors(graph, code)
        codes_set.update(ancestors)
    else:
        # Code not in graph - may be rare/invalid, skip silently
        # print(f"Code {code} not found in concept graph (may be rare/invalid)")
        pass
    
    return codes_set

In [ ]:
# Load OMOPDataset and extract condition codes for all patients
dataset = OMOPDataset(
    root=r"C:\Users\michi\Workspace\mimic-iv-demo-data-in-the-omop-common-data-model-0.9\1_omop_data_csv",
    tables=["condition_occurrence"],
    dataset_name="omop",
    dev=False
)

dataset.stats()

print("Extracting condition codes from all patients...")
        
patient_conditions = defaultdict(list)

# Iterate through all patients
for patient in dataset.iter_patients():
    patient_id = patient.patient_id
    
    # Get all condition events for this patient
    condition_events = patient.get_events(event_type="condition_occurrence")
    
    # Extract condition_concept_id from each event
    for event in condition_events:
        code = event.attr_dict.get("condition_concept_id")
        if code is not None:
            patient_conditions[patient_id].append(code)

print(f"Extracted conditions for {len(patient_conditions)} patients")

In [ ]:
# Filter out conditions codes with <2 occurrences in patient history
filtered_conditions = {}
        
before_count = sum(len(codes) for codes in patient_conditions.values())
before_unique = len(set(code for codes in patient_conditions.values() for code in codes))

for patient_id, codes in patient_conditions.items():
    # Count occurrences of each code
    code_counts = Counter(codes)
    
    filtered_codes = [code for code, count in code_counts.items() if count >= 2]
    
    if filtered_codes:
        filtered_conditions[patient_id] = filtered_codes

after_count = sum(len(codes) for codes in filtered_conditions.values())
after_unique = len(set(code for codes in filtered_conditions.values() for code in codes))

print(f"Before Filtering:")
print(f"Total codes: {before_count}, Unique: {before_unique}")
print("="*70)
print(f"After Filtering:")
print(f"Total codes: {after_count}, Unique: {after_unique}")

In [ ]:
# Initialize N2V with same parameters as KeepEmbedding will use
n2v = N2V(
    path=r"C:\Users\michi\Workspace\mimic-iv-demo-data-in-the-omop-common-data-model-0.9\1_omop_data_csv",
    domain_type=["Condition"],
    embedding_dim=100,
    walk_length=30,
    num_walks=750, 
)

# Build the concept relationship graph from conditions
print("Building concept relationship graph...")
graph = n2v._create_graph()

print(f"Graph loaded successfully:")
print(f"  Nodes (unique concepts): {len(graph.nodes())}")
print(f"  Edges (relationships): {len(graph.edges())}")

print("Applying hierarchy roll-up (dense: each code -> itself + all parents)...")

rolled_up_conditions = {}

# Track statistics
total_original_codes = 0
total_rolled_up_codes = 0
patients_processed = 0

for patient_id, codes in filtered_conditions.items():
    expanded_codes = set()
    
    # For each code, add it and all ancestors
    for code in codes:
        code_and_ancestors = get_code_and_ancestors(graph, code)
        expanded_codes.update(code_and_ancestors)
    
    if expanded_codes:
        rolled_up_conditions[patient_id] = expanded_codes
        total_original_codes += len(codes)
        total_rolled_up_codes += len(expanded_codes)
        patients_processed += 1

# Log statistics
print(f"Roll-up complete:")
print(f"  Patients processed: {patients_processed}")
print(f"  Original codes per patient (avg): {total_original_codes / patients_processed:.2f}")
print(f"  Rolled-up codes per patient (avg): {total_rolled_up_codes / patients_processed:.2f}")
print(f"  Expansion factor: {total_rolled_up_codes / total_original_codes:.2f}x")

In [ ]:
# Construct co-occurrence matrix for rolled-up conditions
print("Building code index from rolled-up conditions...")
        
# Collect all unique codes after roll-up
unique_codes = set()
for codes in rolled_up_conditions.values():
    unique_codes.update(codes)

# Sort for reproducibility
unique_codes = sorted(unique_codes)

# Create bidirectional mapping
code_to_index = {code: idx for idx, code in enumerate(unique_codes)}
index_to_code = {idx: code for code, idx in code_to_index.items()}


print("Building co-occurrence matrix...")

num_codes = len(code_to_index)

# Initialize matrix
cooc_matrix = np.zeros((num_codes, num_codes), dtype=np.float32)

# Track statistics
total_pairs = 0
patients_processed = 0

for patient_id, codes in rolled_up_conditions.items():
    # Convert codes to indices
    code_indices = [code_to_index[code] for code in codes]
    
    # Create all unique pairs
    for i in range(len(code_indices)):
        for j in range(i + 1, len(code_indices)):
            idx_i = code_indices[i]
            idx_j = code_indices[j]
            
            # Increment both matrix[i,j] and matrix[j,i] for symmetry
            cooc_matrix[idx_i, idx_j] += 1.0
            cooc_matrix[idx_j, idx_i] += 1.0
            
            total_pairs += 1
    
    patients_processed += 1

print(f"Matrix construction complete:")
print(f"  Matrix shape: {cooc_matrix.shape}")
print(f"  Patients processed: {patients_processed}")
print(f"  Total co-occurrence pairs: {total_pairs}")
print(f"  Matrix sparsity: {(cooc_matrix == 0).sum() / cooc_matrix.size * 100:.2f}%")
print(f"  Matrix sum (total co-occurrences): {cooc_matrix.sum():.0f}")
print(f"  Min value: {cooc_matrix.min():.2f}, Max value: {cooc_matrix.max():.2f}")

In [ ]:
# Parameters for GloveDataset (standard GloVe hyperparameters)
x_max = 100  # Maximum co-occurrence count before weight saturates
alpha = 0.75  # Weighting factor exponent
batch_size = 128

# Create GloveDataset from co-occurrence matrix
print("Creating GloveDataset from co-occurrence matrix...")
num_words = cooc_matrix.shape[0]
glove_dataset = GloveDataset(cooc_matrix, num_words, x_max, alpha)

print(f"  GloveDataset created:")
print(f"    Num words (diagnosis codes): {num_words}")
print(f"    Dataset size (co-occurrence pairs): {len(glove_dataset)}")

# Create DataLoader
data_loader = DataLoader(glove_dataset, batch_size=batch_size, shuffle=True)
print(f"  DataLoader created: batch_size={batch_size}, num_batches={len(data_loader)}")

# Initialize KeepEmbedding with MATCHING parameters as builder
print("\nInitializing KeepEmbedding with builder parameters...")
keep_model = KeepEmbedding(
    dataset=None,  # GloVe training doesn't require full dataset
    graph = graph,  # Pass the concept relationship graph for Node2Vec
    num_words=num_words,
    embedding_dim=100,
    lambda_reg=1.0,  # Regularization strength (balances GloVe vs graph prior)
    reg_norm=None,  # Use cosine similarity for regularization distance
    log_scale=False,  # No log scaling on regularization
    device=device
)

print(f"  KeepEmbedding initialized:")
print(f"    Embedding dimension: 100")
print(f"    Regularization lambda: 1.0")
print(f"    Device: {device}")
print(f"\nModel ready for training!")

In [ ]:
print("Training KeepEmbedding model with GloVe objective + graph regularization...")
print(f"  Total batches: {len(data_loader)}")
print(f"  Training for 50 epochs\n")

trainer = Trainer(model=keep_model)

# Train with GloVe objective + Node2Vec graph regularization
trainer.train(
    train_dataloader=data_loader,
    val_dataloader=None,  # No validation split for embedding training
    epochs=50,
    optimizer_class=torch.optim.Adam,
    learning_rate=0.01,
    monitor="loss",  # Monitor training loss
    patience=10,  # Early stopping patience
)

print("\nTraining complete!")
print(f"Learned embeddings shape: {keep_model.embeddings_v.weight.shape}")
print(f"Ready for downstream tasks!")